In [1]:
import sys
import pandas as pd
import logging
sys.path.append('/Users/BKieft/Metabolomics/metatlas2/metatlas2')
import database_interact as dbi
import pubchem_retrieval as pcr
import load_tools as ldt
import lcmsruns_tools as lrt
import ms1_ms2_analysis as msa
import rt_align_tools as rat
import targeted_analysis as tga
import targeted_gui as tgi
import logging_config as lcf

logger = lcf.setup_logging(log_level=logging.INFO)
pd.options.display.max_colwidth = 300

In [2]:
# Required: load config file
CONFIG = ldt.load_metatlas_config("/Users/BKieft/Metabolomics/metatlas2/configs/metatlas2_config.yaml")
# Required: table with compounds to add to db. Minimally, must have columns 'inchi_key' and 'label'
COMPOUND_INPUTS = ["/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_QC_POS.tsv",
                   "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_POS.tsv", 
                   "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_NEG.tsv",
                   "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_EMA_POS.tsv",
                   "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_EMA_NEG.tsv"]
# Required: table of compounds; minimally, must have columns 'inchi_key', 'chromatography', 'polarity', 'rt_peak', 'mz', 'adduct'
ATLAS_INPUTS = ["/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_POS.tsv",
                "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_QC_POS.tsv"]
# Required: name of atlas (does not have to be unique) 
ATLAS_NAMES = ["HILIC Positive ISTD Default", "HILIC Positive QC Default"]
# Optional: description of atlas
ATLAS_DESC = [f"Default ISTD compounds for HILIC Positive", f"Default QC compounds for HILIC Positive"]
# Required: Project name
PROJECT_NAME = "20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519"
# Required: paths based on config
PROJECT_FILES_PATH = f"{CONFIG['paths']['projects_dir']}/{PROJECT_NAME}/lcmsruns"
PROJECT_DB_PATH = f"{CONFIG['paths']['projects_dir']}/{PROJECT_NAME}/{PROJECT_NAME}.duckdb"
# Required: path to id report
ANALYSIS_OUTPUT_PATH = f'/Users/BKieft/Downloads/{PROJECT_NAME}'

In [3]:
new_main_database = False
new_atlases = False
new_rt_alignment = False
new_analysis = True

In [4]:
if new_main_database is True:
    dbi.create_metatlas_database(CONFIG)
    for COMPOUNDS_INPUT in COMPOUND_INPUTS:
        compounds_df = ldt.load_compound_input(COMPOUNDS_INPUT)
        pcr.retrieve_pubchem_info(compounds_df, CONFIG)
        dbi.add_compounds_to_db(compounds_df, CONFIG, COMPOUNDS_INPUT)
    dbi.validate_database(CONFIG)

In [5]:
if new_atlases is True:
    new_atlas_info = ()
    for ATLAS_INPUT, ATLAS_NAME, ATLAS_DESC in zip(ATLAS_INPUTS, ATLAS_NAMES, ATLAS_DESC):
        atlas_compounds_df = ldt.load_atlas_input(ATLAS_INPUT)
        atlas_uid, atlas_name = dbi.create_atlas_from_compounds(atlas_compounds_df, ATLAS_NAME, ATLAS_DESC, CONFIG)
        new_atlas_info += ((atlas_uid, atlas_name),)
    dbi.validate_database(CONFIG)
    QC_ATLAS_UID = next(uid for uid, name in new_atlas_info if "QC" in name)
    TARGET_ATLAS_UID = next(uid for uid, name in new_atlas_info if "QC" not in name)
else:
    QC_ATLAS_UID = "atl-raw-7906f4bf395d4ed791f3763ed10f61c8"
    TARGET_ATLAS_UID = "atl-raw-2e24a9969a6a4758844b99162e9e0a03"
    logger.info(f"Skipping new atlas creation and using QC Atlas: {QC_ATLAS_UID} and TARGET Atlas: {TARGET_ATLAS_UID}")

2025-09-04 19:50:02 | metatlas2 | INFO | Skipping new atlas creation and using QC Atlas: atl-raw-7906f4bf395d4ed791f3763ed10f61c8 and TARGET Atlas: atl-raw-2e24a9969a6a4758844b99162e9e0a03


In [6]:
if new_rt_alignment is True:
    dbi.create_project_database(PROJECT_DB_PATH, CONFIG)
    lcmsrun_files = dbi.save_lcmsruns_to_db(PROJECT_DB_PATH, PROJECT_NAME, PROJECT_FILES_PATH, CONFIG)
    matches_df, matching_stats = msa.extract_and_match_qc_compounds(PROJECT_DB_PATH, QC_ATLAS_UID, CONFIG)
    best_model, model_results, model_stats = rat.build_rt_alignment_model(matches_df, CONFIG)
    ANALYSIS_ATLAS_UID = rat.apply_rt_correction_to_target(PROJECT_DB_PATH, TARGET_ATLAS_UID, CONFIG, best_model, lcmsrun_files, model_results)
else:
    ANALYSIS_ATLAS_UID = "atl-rta-0c89e582233e481ca4f1920611f6007f"
    logger.info(f"Skipping new RT alignment and using {ANALYSIS_ATLAS_UID}")

2025-09-04 19:50:02 | metatlas2 | INFO | Skipping new RT alignment and using atl-rta-0c89e582233e481ca4f1920611f6007f


In [7]:
if new_analysis is True:
    atlas_dataframe, project_data, plot_data = tga.run_targeted_analysis_workflow(PROJECT_DB_PATH, ANALYSIS_ATLAS_UID, CONFIG)
    gui = tgi.create_gui(plot_data, CONFIG, PROJECT_DB_PATH)
    gui._project_data = project_data
    display(gui)

2025-09-04 19:50:02 | metatlas2.targeted_analysis | INFO | Setting up targeted analysis database...
2025-09-04 19:50:02 | metatlas2.targeted_analysis | INFO | Running fresh targeted analysis...
2025-09-04 19:50:02 | metatlas2.targeted_analysis | INFO | Loading target atlas...
2025-09-04 19:50:02 | metatlas2.targeted_analysis | INFO | Running fresh targeted analysis...
2025-09-04 19:50:02 | metatlas2.targeted_analysis | INFO | Loading target atlas...
2025-09-04 19:50:02 | metatlas2.database_interact | INFO | Retrieved 65 compounds from project database for atlas: HILIC Positive ISTD Default (RT Corrected) (atl-rta-0c89e582233e481ca4f1920611f6007f)
2025-09-04 19:50:02 | metatlas2.database_interact | INFO | RT-corrected atlas detected with experimental data
2025-09-04 19:50:02 | metatlas2.database_interact | INFO | Retrieved 65 compounds from project database for atlas: HILIC Positive ISTD Default (RT Corrected) (atl-rta-0c89e582233e481ca4f1920611f6007f)
2025-09-04 19:50:02 | metatlas2.da

Processing files in parallel:   0%|          | 0/6 [00:00<?, ?it/s]

2025-09-04 19:50:09 | metatlas2.ms1_ms2_analysis | INFO | Extraction complete:
2025-09-04 19:50:09 | metatlas2.ms1_ms2_analysis | INFO |   Total compounds: 60
2025-09-04 19:50:09 | metatlas2.ms1_ms2_analysis | INFO |   Compounds with EIC data: 60
2025-09-04 19:50:09 | metatlas2.ms1_ms2_analysis | INFO |   Compounds with MS2 data: 38
2025-09-04 19:50:09 | metatlas2.ms1_ms2_analysis | INFO |   Compounds with MS2 hits: 37
2025-09-04 19:50:09 | metatlas2.ms1_ms2_analysis | INFO |   Total EIC traces: 284
2025-09-04 19:50:09 | metatlas2.ms1_ms2_analysis | INFO |   Total MS2 spectra: 352
2025-09-04 19:50:09 | metatlas2.ms1_ms2_analysis | INFO |   Total compounds: 60
2025-09-04 19:50:09 | metatlas2.ms1_ms2_analysis | INFO |   Compounds with EIC data: 60
2025-09-04 19:50:09 | metatlas2.ms1_ms2_analysis | INFO |   Compounds with MS2 data: 38
2025-09-04 19:50:09 | metatlas2.ms1_ms2_analysis | INFO |   Compounds with MS2 hits: 37
2025-09-04 19:50:09 | metatlas2.ms1_ms2_analysis | INFO |   Total EI

In [8]:
# Test to verify MS2 scores are being calculated correctly
if new_analysis is True:
    # Check a few compounds to see if they have scores
    test_compounds = list(plot_data.keys())[:5]
    
    for inchi_key in test_compounds:
        compound_data = plot_data[inchi_key]
        best_ms2 = compound_data.get('best_ms2', {})
        ms2_data = compound_data.get('ms2_data', {})
        
        print(f"\nCompound: {compound_data['original_atlas_data']['compound_name']}")
        print(f"  InChI Key: {inchi_key}")
        
        # Handle None scores properly
        score = best_ms2.get('score', None)
        print(f"  Best MS2 Score: {score if score is not None else 'N/A'}")
        print(f"  Best MS2 Num Matches: {best_ms2.get('num_matches', 'N/A')}")
        print(f"  Selection Method: {best_ms2.get('selection_method', 'N/A')}")
        print(f"  Files with MS2 data: {len(ms2_data)}")
        
        # Check raw class data and test similarity calculation directly
        compound_class_data = project_data.get_compound(inchi_key)
        if compound_class_data:
            spectra_with_hits = compound_class_data.get_spectra_with_hits()
            print(f"  Raw class data - spectra with hits: {len(spectra_with_hits)}")
            if spectra_with_hits:
                best_spectrum = max(spectra_with_hits, key=lambda s: s.best_hit.score)
                print(f"  Raw class data - best hit score: {best_spectrum.best_hit.score}")
                
                # Test direct similarity calculation
                if len(best_spectrum.hits) > 0:
                    hit = best_spectrum.hits[0]
                    test_score = sph.calculate_cosine_similarity(
                        best_spectrum.mz_values, 
                        best_spectrum.intensity_values,
                        hit.ref_mz_values,
                        hit.ref_intensity_values
                    )
                    print(f"  Direct similarity test: {test_score}")
        
        # Check if this compound has a valid score (not None and > 0)
        if score is not None and score > 0:
            print(f"  ✓ Found compound with valid score: {score}")
            break  # Found one with a score, that's enough for testing
    else:
        print("\n⚠️  No compounds found with valid MS2 scores")
        print("This might indicate an issue with:")
        print("  - MS2 reference database loading")
        print("  - Spectrum similarity calculation")
        print("  - Data flow from class objects to GUI format")
        
        # Test the similarity calculation function directly
        print("\nTesting similarity calculation function...")
        import spectrum_handlers as sph
        
        # Create test spectra
        test_query_mz = np.array([100, 200, 300])
        test_query_int = np.array([1000, 500, 2000])
        test_ref_mz = np.array([100.001, 200.002, 300.003])  # Very close matches
        test_ref_int = np.array([900, 600, 1800])
        
        test_score = sph.calculate_cosine_similarity(
            test_query_mz, test_query_int, test_ref_mz, test_ref_int
        )
        print(f"Test similarity score (should be > 0.9): {test_score}")


Compound: ABMBA (unlabeled)
  InChI Key: LCMZECCEEOQWLQ-UHFFFAOYSA-N
  Best MS2 Score: N/A
  Best MS2 Num Matches: None
  Selection Method: none
  Files with MS2 data: 0
  Raw class data - spectra with hits: 0

Compound: nicotinamide (U - 13C)
  InChI Key: DFPAKSUCGFBDDF-IDEBNGHGSA-N
  Best MS2 Score: 0.0
  Best MS2 Num Matches: 17
  Selection Method: reference_hit
  Files with MS2 data: 4
  Raw class data - spectra with hits: 9
  Raw class data - best hit score: 0.0


NameError: name 'sph' is not defined

In [ ]:
#plot_data['AGPKZVBTJJNPAG-WHFBIAKZSA-N']['ms2_data']

{'20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_POS_MS2_36_Fall-VFa_7_Rg70to1050-CE102040norm-veg-core-S1_Run136.h5': {'num_ms2_entries': 5,
  'num_hits': 160,
  'ms2_entries': [{'inchi_key': 'AGPKZVBTJJNPAG-WHFBIAKZSA-N',
    'spectrum': [[46.386878967285156,
      68.86904907226562,
      81.0428695678711,
      86.09579467773438,
      91.83330535888672,
      98.47689819335938,
      132.10113525390625],
     [9331.201171875,
      8899.521484375,
      10113.853515625,
      150956.203125,
      8366.1884765625,
      10194.142578125,
      86648.0390625]],
    'intensity_peak': 150956.203125,
    'rt': 9.40108585357666,
    'precursor_mz': 132.10183715820312,
    'filename': '20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_POS_MS2_36_Fall-VFa_7_Rg70to1050-CE102040norm-veg-core-S1_Run136.h5'},
   {'inchi_key': 'AGPKZVBTJJNPAG-WHFBIAKZSA-N',
    'spectrum': [[44.0494384765625,
      75.23883056640625,
      75.93054962158203,
      77.39296722412

In [8]:
inchi_key = 'AGPKZVBTJJNPAG-WHFBIAKZSA-N'
mods = gui.get_modifications()
print(mods.get_rt_bounds(inchi_key))
print(mods.get_annotations(inchi_key))

None
{}


In [9]:
if new_analysis is True:
    post_analysis_atlas_uid, targeted_analysis_uid, comprehensive_report = tga.run_post_analysis_workflow(
        project_db_path=PROJECT_DB_PATH,
        analysis_atlas_uid=ANALYSIS_ATLAS_UID,
        gui_container=gui,
        atlas_dataframe=atlas_dataframe,
        project_name=PROJECT_NAME,
        config=CONFIG,
        analysis_output_path=ANALYSIS_OUTPUT_PATH
    )

2025-09-04 19:43:05 | metatlas2.targeted_analysis | INFO | Starting post-analysis workflow...
2025-09-04 19:43:05 | metatlas2.targeted_analysis | INFO | Creating post-analysis atlas...
2025-09-04 19:43:05 | metatlas2.simple_cache | INFO | GUI cache (complete) saved to /Users/BKieft/Metabolomics/metatlas2/data/test_data/projects/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519/cache/gui_cache/complete/2025-09-04T19-43-05-230815
2025-09-04 19:43:05 | metatlas2.targeted_analysis | INFO | Saved final AnalystModifications to complete cache: 2025-09-04T19:43:05.230815
2025-09-04 19:43:05 | metatlas2.targeted_analysis | INFO | Creating post-analysis atlas...
2025-09-04 19:43:05 | metatlas2.simple_cache | INFO | GUI cache (complete) saved to /Users/BKieft/Metabolomics/metatlas2/data/test_data/projects/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519/cache/gui_cache/complete/2025-09-04T19-43-05-230815
2025-09-04 19:43:05 | metatlas2.targeted_analysis | INFO | Sav

Processing compounds:   0%|          | 0/60 [00:00<?, ?it/s]

2025-09-04 19:43:05 | metatlas2.database_interact | INFO | Report saved to /Users/BKieft/Downloads/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519
2025-09-04 19:43:05 | metatlas2.targeted_analysis | INFO | Generated comprehensive report and saved to /Users/BKieft/Downloads/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519
2025-09-04 19:43:05 | metatlas2.targeted_analysis | INFO | Post-analysis workflow completed successfully
2025-09-04 19:43:05 | metatlas2.targeted_analysis | INFO | Generated comprehensive report and saved to /Users/BKieft/Downloads/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519
2025-09-04 19:43:05 | metatlas2.targeted_analysis | INFO | Post-analysis workflow completed successfully


In [10]:
dbi.validate_database(CONFIG, PROJECT_DB_PATH)

2025-09-04 19:43:05 | metatlas2.database_interact | INFO | Database Validation:
2025-09-04 19:43:05 | metatlas2.database_interact | INFO |    Atlases: 4
2025-09-04 19:43:05 | metatlas2.database_interact | INFO |    Targeted analyses: 3
2025-09-04 19:43:05 | metatlas2.database_interact | INFO |    RT alignment models: 1
2025-09-04 19:43:05 | metatlas2.database_interact | INFO |    Experimental RT/MZ entries: 65
2025-09-04 19:43:05 | metatlas2.database_interact | INFO |    Available atlases:
2025-09-04 19:43:05 | metatlas2.database_interact | INFO |       atl-rta-0c89e582233e481ca4f1920611f6007f
2025-09-04 19:43:05 | metatlas2.database_interact | INFO |       atl-tga-a9e7fd0ab47f43c69e54d4248b5a4524
2025-09-04 19:43:05 | metatlas2.database_interact | INFO |       atl-tga-d682e64f7145441593a2efd38e5ef480
2025-09-04 19:43:05 | metatlas2.database_interact | INFO |       atl-tga-5957c5649496448c9480b10032eb4247
2025-09-04 19:43:05 | metatlas2.database_interact | INFO |    Targeted analysis e